In [1]:
import pandas as pd
from transformers import AutoTokenizer
from transformers import AutoTokenizer
from collections import defaultdict
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import torch
import torch
import torch.nn as nn
import math
import torch
from torch.nn.utils.rnn import pad_sequence
import torch.optim as optim
import os
import tqdm 

/Users/maryamsaad/Documents/arabic_poetry_generator/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

df=pd.read_csv("/Users/maryamsaad/Documents/arabic_poetry_generator/datasets/cleaned_v2_poetry_dataset.csv",encoding='utf-8')

In [3]:
df[:2]

,Unnamed: 0,العصر,البيت,الشاعر,clean_text,word_count,word_count_bin
0,94,الحديث,هَمَّتِ الفُلكُ وَاِحتَواها الماءُ وَحَداها...,أحمد شوقي,همت الفلك واحتواها الما وحداها بمن تقل الرجا,8,"(5, 10]"
1,95,الحديث,ضَرَبَ البَحرُ ذو العُبابِ حَوالَي ها سَماء...,أحمد شوقي,ضرب البحر ذو العباب حوالي ها سما قد اكبرتها السما,10,"(5, 10]"


## Byte pair tokenization

In [26]:
class BPETokenizer:
    def __init__(self, model_name):
        self.tokenizer = AutoTokenizer.from_pretrained("konstantindobler/mistral7b-ar-tokenizer-swap-pure-bf16")
    
    def encode(self, text):
        return self.tokenizer.encode(text, add_special_tokens=True)
    
    def get_token_frequencies(self, texts):
        token_freqs = defaultdict(int)
        for text in texts:
            tokens = self.encode(text)
            for token in tokens:
                token_freqs[token] += 1
        return token_freqs
    


In [27]:
# Example usage:
df = pd.read_csv("/Users/maryamsaad/Documents/arabic_poetry_generator/datasets/cleaned_v2_poetry_dataset.csv", encoding='utf-8')
bpe_tokenizer = BPETokenizer("konstantindobler/mistral7b-ar-tokenizer-swap-pure-bf16")
texts = df['clean_text'].dropna().astype(str)
encoded_texts = texts.apply(bpe_tokenizer.encode)
token_freqs = bpe_tokenizer.get_token_frequencies(texts)
vocab_size = bpe_tokenizer.tokenizer.vocab_size  # Always use the tokenizer's vocab_size
print(f"Vocabulary Size: {vocab_size}")

Vocabulary Size: 32768


## Build model

In [6]:
batch_size = 16

In [53]:

train_texts, val_texts = train_test_split(df['clean_text'].dropna().astype(str), test_size=0.1, random_state=42)

In [10]:
def load_data(encoded_dataset, batch_size=16):
    dataset = TokenDataset(encoded_dataset)
    dataloader = torch.utils.data.DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        collate_fn=collate_fn 
    )
    return dataloader
def collate_fn(batch):
    input_ids = [item['input_ids'] for item in batch]
    padded = pad_sequence(input_ids, batch_first=True, padding_value=0)
    return {'input_ids': padded}

class TokenDataset(torch.utils.data.Dataset):
    def __init__(self, token_lists):
        self.token_lists = token_lists
    def __len__(self):
        return len(self.token_lists)
    def __getitem__(self, idx):
        tokens = self.token_lists[idx]
        return {'input_ids': torch.tensor(tokens, dtype=torch.long)}
 

In [11]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [12]:
class FeedForward(nn.Module):
    def __init__(self, d_model, dim_ff, dropout):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.layers(x)


In [13]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, nhead, dim_ff, dropout):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=True
        )

        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, dim_ff, dropout)


    def forward(self, x, attn_mask=None, key_padding_mask=None):
        # Self-Attention block
        h = self.ln1(x)
        attn_out, _ = self.self_attn(
            h, h, h,
            attn_mask=attn_mask,
            key_padding_mask=key_padding_mask
        )
        x = x + attn_out

        # Feed-Forward block
        h = self.ln2(x)
        x = x + self.ff(h)

        return x



In [14]:
class TP_PoetDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, num_layers=6, dim_ff=2048, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([DecoderBlock(d_model, nhead, dim_ff, dropout) for _ in range(num_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, attn_mask=None, padding_mask=None):
        x = self.token_embedding(input_ids) * math.sqrt(self.token_embedding.embedding_dim)
        x = self.positional_encoding(x)
        for layer in self.layers:
            x = layer(x, attn_mask=attn_mask, key_padding_mask=padding_mask)
        return self.lm_head(self.final_norm(x))


In [13]:
vocab_size = bpe_tokenizer.tokenizer.vocab_size
print("Vocab size:", vocab_size)

Vocab size: 32768


In [16]:
# model = TP_PoetDecoder(
#     vocab_size=vocab_size,
#     d_model=256,
#     nhead=4,
#     num_layers=3,
#     dim_ff=512,
#     dropout=0.2
# )
model = TP_PoetDecoder(
    vocab_size=32768,
    d_model=256,
    nhead=4,
    num_layers=3,
    dim_ff=512,
    dropout=0.2
)

In [54]:
train_encoded = train_texts.apply(bpe_tokenizer.encode).tolist()
train_dataloader = load_data(train_encoded, batch_size=batch_size)

NameError: name 'batch_size' is not defined

In [16]:
def train_function(model, dataloader, num_epochs, vocab_size):
    checkpoint_dir='checkpoints'
    os.makedirs(checkpoint_dir, exist_ok=True)
    model.to(device='mps')
    model.train()
    criterion = torch.nn.CrossEntropyLoss(ignore_index=0)
    optimizer = optim.Adam(model.parameters(), lr=0.0001)
    
    for epoch in tqdm.tqdm(range(num_epochs)):
        total_loss = 0
        for batch_idx, batch in enumerate(dataloader):
            input_ids = batch['input_ids'].to('mps') 
            inputs = input_ids[:, :-1]
            targets = input_ids[:, 1:]
            optimizer.zero_grad()
            logits = model(inputs)
            loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1))
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1} completed. Average Loss: {avg_loss:.4f}")

        checkpoint_path = os.path.join(checkpoint_dir, f"model_v2_epoch_{epoch+1}.pt")
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": avg_loss
        }, checkpoint_path)
        print(f"Checkpoint saved: {checkpoint_path}")


In [17]:
train_function(model, train_dataloader, num_epochs=8, vocab_size=vocab_size)

  0%|          | 0/12 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch 1 completed. Average Loss: 4.9406


  8%|▊         | 1/12 [36:21<6:39:58, 2181.65s/it]

Checkpoint saved: checkpoints/model_epoch_1.pt
Epoch 2 completed. Average Loss: 2.6134


 17%|█▋        | 2/12 [1:16:53<6:28:06, 2328.63s/it]

...

Checkpoint saved: checkpoints/model_epoch_5.pt
Epoch 6 completed. Average Loss: 0.2979


 50%|█████     | 6/12 [4:49:07<4:44:54, 2849.14s/it]

Checkpoint saved: checkpoints/model_epoch_6.pt


 50%|█████     | 6/12 [5:12:52<5:12:52, 3128.73s/it]


KeyboardInterrupt: 

In [33]:
model = TP_PoetDecoder(
    vocab_size=32768,
    d_model=512,   # 512 ÷ 8 = 64
    nhead=8,
    num_layers=6,
    dim_ff=2048,
    dropout=0.1
)
checkpoint = torch.load("checkpoints/model_epoch_6.pt", map_location="mps")
model.load_state_dict(checkpoint["model_state_dict"])
model.to('mps')
print(f"Model loaded from epoch {checkpoint['epoch']} with loss {checkpoint['loss']:.4f}")

Model loaded from epoch 6 with loss 0.2979


In [ ]:
train_dataloader = load_data(train_encoded, batch_size=16)
train_encoded = train_texts.apply(bpe_tokenizer.encode).tolist()
val_dataloader = load_data(val_texts, batch_size=16)
def test_model(model, dataloader_dataset):
    with torch.no_grad():
        model.eval()
        total_loss = 0
        criterion = torch.nn.CrossEntropyLoss(ignore_index=0) 
        for batch_idx, batch in enumerate(dataloader_dataset):
            input_ids = batch['input_ids'].to('mps')  
            inputs = input_ids[:, :-1]
            targets = input_ids[:, 1:]
            logits = model(inputs)
            loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1))
            total_loss += loss.item()
        
        avg_loss = total_loss / len(dataloader_dataset)
        print(f"Test Average Loss: {avg_loss:.4f}")

test_model(model,val_dataloader,batch_size=16)    


In [47]:
def generate(model, input_ids, max_new_tokens=10, temperature=0.9,
             top_k=10,  repetition_penalty=1.9):

    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):

            logits = model(input_ids)[:, -1, :]

            # repetition penalty
            for token in set(input_ids.flatten().tolist()):
                logits[:, token] /= repetition_penalty

            # temperature
            logits = logits / temperature

            topk_vals, topk_idx = torch.topk(logits, top_k)  # [batch_size, top_k]
            probs = torch.softmax(topk_vals, dim=-1)         # [batch_size, top_k]
            sampled_idx = torch.multinomial(probs, num_samples=1)  # [batch_size, 1]
            next_token = torch.gather(topk_idx, 1, sampled_idx)    # [batch_size, 1]
            input_ids = torch.cat([input_ids, next_token], dim=1)
            generated_text = bpe_tokenizer.tokenizer.decode(input_ids[0].tolist())

    return generated_text
sentence="هي حلم الهوي"
encoded_input = bpe_tokenizer.encode(sentence)
input_tensor = torch.tensor(encoded_input, dtype=torch.long).unsqueeze(0).to('mps')
generate(model, input_tensor, max_new_tokens=10)

'<s> هي حلم الهوي الجوز جوار نفوس علوم اساس علوم الاثار الاثار نهار عهد'

In [48]:
sentence="ذهب وليد "
encoded_input = bpe_tokenizer.encode(sentence)
input_tensor = torch.tensor(encoded_input, dtype=torch.long).unsqueeze(0).to('mps')
generate(model, input_tensor, max_new_tokens=10)

'<s> ذهب وليد امجد المشرق سحب نع نع عي يه لحوض'

In [49]:
sentence="ي صمت الليل زهب"
encoded_input = bpe_tokenizer.encode(sentence)
input_tensor = torch.tensor(encoded_input, dtype=torch.long).unsqueeze(0).to('mps')
generate(model, input_tensor, max_new_tokens=10)

'<s> ي صمت الليل زهب البحر تسبق والكل مينيس النار والوجه باسمه الخمر القران'

In [50]:
sentence="ذهب اليل و طلع الفجر"
encoded_input = bpe_tokenizer.encode(sentence)
input_tensor = torch.tensor(encoded_input, dtype=torch.long).unsqueeze(0).to('mps')
generate(model, input_tensor, max_new_tokens=10)

'<s> ذهب اليل و طلع الفجر مستقبلا الامر مرهفا النجم البعيد سليم الرياح العقاب ربه'

In [51]:
sentence="لا قافية اليوم سوى انفاسي"
encoded_input = bpe_tokenizer.encode(sentence)
input_tensor = torch.tensor(encoded_input, dtype=torch.long).unsqueeze(0).to('mps')
generate(model, input_tensor, max_new_tokens=10)

'<s> لا قافية اليوم سوى انفاسي تتكيا وحدهاصن واثق اجملهوالعاظم'